### Learning Instruction Finetuning

In [ ]:
%load_ext autoreload
%load_ext tensorboard
%autoreload 2


In [2]:
from pathlib import Path
import os
import socket
import sys
from dotenv import load_dotenv
from pathlib import Path
import sys

In [3]:
try:
    project_root = next(p for p in Path().cwd().resolve().parents if (p /"env.env").exists())
except StopIteration:
    # Fallback to current working directory if not found
    project_root = Path.cwd()

sys.path.insert(0, str(project_root))

print(f"Project Root: {project_root}")
load_dotenv((project_root /'env.env').resolve())
data_dir = project_root / "datasets"

Project Root: /storage/ice1/1/0/vchopra37/projects/learning_env


In [4]:
# Check if running on the Pace server
hostname = socket.gethostname()
IS_PACE_SERVER = "pace" in hostname.lower()  # Replace 'pace' with a unique keyword from your server's hostname
IS_PACE_SERVER

True

In [5]:
import torch
from datasets import Dataset as hf_data
from torch.utils.data import Dataset
import tiktoken
import numpy, tqdm
import matplotlib.pyplot as plt
%matplotlib inline

In [6]:
## Accelerator for Pytorch
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
device

'cuda'

In [7]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

In [8]:

from model_training.framework.data.post_train.instruction_tuning.raw_data.simple_example.instruct_data import download_and_load_file
from model_training.framework.data.post_train.instruction_tuning.prompt_style.styles import SFTDatasetPrepare, AlpacaFormatOld
from model_training.framework.data.post_train.dataloader import InstructionDataset, custom_collate_fn, create_instruction_dataloader

    ## Fix this import
    
    # raise Exception("Not Running on Pace")
        # from src.model import GPT

In [9]:
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(i, props.name, f"{props.total_memory / 1024**3:.1f} GiB")

GPU count: 1
0 NVIDIA A100-PCIE-40GB 39.5 GiB


### Step-1:-  Preparing the dataset

We are using Alpaca Format, every models have their own format, so the format of the tuning can differ. <br>
THis format is called ALPACA format{instruction, input output}... <br>
 There are different formats that exist and the data format doesn't matter, we need to take prompt style templates for instruction tuning. We need to apply those prompt templates for instruction tuning. 

In [10]:
file_name = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)
data = download_and_load_file(data_dir=(data_dir / "instruction_tuning"), file_name=file_name, url = url)
print("Number of entries:", len(data))
data[50] , data[999]

Number of entries: 1100


({'instruction': 'Identify the correct spelling of the following word.',
  'input': 'Ocassion',
  'output': "The correct spelling is 'Occasion.'"},
 {'instruction': "What is an antonym of 'complicated'?",
  'input': '',
  'output': "An antonym of 'complicated' is 'simple'."})

In [11]:
# alpaca = AlpacaFormatOld(data)

In [12]:
# ## Example of format
# print(alpaca.format_input(instruction=data[50]['instruction'], in_text=data[50]['input']))
# print("=============================================")
# print(alpaca.detailed_response(out_text = data[50]['output']))

In [13]:
# sample = (alpaca.complete_response(instruction=data[999]['instruction'], in_text=data[999]['input'], out_text=data[999]['output']))
# print(sample)

In [14]:
dataset  = hf_data.from_list(data)
sft_data= SFTDatasetPrepare(model_name=MODEL_NAME, template="model",)
prepared = sft_data.prepare(dataset)

Map:   0%|          | 0/1100 [00:00<?, ? examples/s]

In [15]:
print(prepared[0])
print(sft_data.preview(prepared[0]))

{'prompt': [{'content': 'Evaluate the following phrase by transforming it into the spelling given.\n\nInput:\nfreind --> friend', 'role': 'user'}], 'completion': [{'content': 'The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".', 'role': 'assistant'}]}
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Evaluate the following phrase by transforming it into the spelling given.

Input:
freind --> friend<|im_end|>
<|im_start|>assistant
The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".<|im_end|>



#### Spliting the data 

In [16]:
train_portion = int(len(data) * 0.85)  # 85% for training
test_portion = int(len(data) * 0.1)    # 10% for testing
val_portion = len(data) - train_portion - test_portion  # Remaining 5% for validation

train_data = prepared.select(range(train_portion))
test_data = prepared.select(
    range(train_portion, train_portion + test_portion)
)
val_data = prepared.select(
    range(train_portion + test_portion, len(prepared))
)

print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))

Training set length: 935
Validation set length: 55
Test set length: 110


### Step-2: - Prepare the Training Set Batches

#### Setting up Tokenizer

In [17]:
# tokenizer = tiktoken.get_encoding("gpt2")
# pad_token_id= tokenizer.encode('<|endoftext|>', allowed_special={"<|endoftext|>"})[0]
# pad_token_id
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

pad_token_id = tokenizer.pad_token_id
pad_token_id

151643

In [18]:
train_dataset = InstructionDataset(
    ds=train_data,
    tokenizer=tokenizer,
    prompt_style=sft_data,
)


In [19]:
val_dataset = InstructionDataset(
    ds=val_data,
    tokenizer=tokenizer,
    prompt_style=sft_data,
)
test_dataset = InstructionDataset(
    ds=test_data,
    tokenizer=tokenizer,
    prompt_style=sft_data,
)

#### Sample Data Prepare

In [20]:

inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]

batch = (
    inputs_1,
    inputs_2,
    inputs_3
)
custom_collate_fn(batch, pad_token_id=pad_token_id)

(tensor([[     0,      1,      2,      3,      4],
         [     5,      6, 151643, 151643, 151643],
         [     7,      8,      9, 151643, 151643]]),
 tensor([[     1,      2,      3,      4, 151643],
         [     6, 151643,   -100,   -100,   -100],
         [     8,      9, 151643,   -100,   -100]]))

In [21]:
logits_1 = torch.tensor(
    [[-1.0, 1.0],  # 1st training example
     [-0.5, 1.5]]  # 2nd training example
)
targets_1 = torch.tensor([0, 1])


loss_1 = torch.nn.functional.cross_entropy(logits_1, targets_1)
print(loss_1)

logits_2 = torch.tensor(
    [[-1.0, 1.0],
     [-0.5, 1.5],
     [-0.5, 1.5]]  # New 3rd training example
)
targets_2 = torch.tensor([0, 1, 1])

loss_2 = torch.nn.functional.cross_entropy(logits_2, targets_2)
print(loss_2)
targets_3 = torch.tensor([0, 1, -100])

loss_3 = torch.nn.functional.cross_entropy(logits_2, targets_3)
print(loss_3)
print("loss_1 == loss_3:", loss_1 == loss_3)

tensor(1.1269)
tensor(0.7936)
tensor(1.1269)
loss_1 == loss_3: tensor(True)


### Creating, Training and Testing Dataset

In [22]:
from functools import partial
# Changing the defaults at runtime
custom_collate_fn1 = partial(custom_collate_fn, device=device, allowed_max_length = 1024, pad_token_id=pad_token_id)

In [23]:
from torch.utils.data import DataLoader


num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_loader = create_instruction_dataloader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=custom_collate_fn1,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)
val_loader = create_instruction_dataloader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=custom_collate_fn1,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

test_loader = create_instruction_dataloader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=custom_collate_fn1,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

In [24]:
print("Train loader:")
for inputs, targets in train_loader:
    sample_input , sample_target = inputs, targets
    print(inputs.shape, targets.shape)
    break

Train loader:
torch.Size([8, 59]) torch.Size([8, 59])


### Step-4: - Loading the Model Testing inference

In [25]:
from model_training.framework.model.load_model.gpt2.model import return_model
from model_training.framework.model.load_model.gpt2.configs import load_model_config
from model_training.framework.model.components.inference.inference import gen_sample
from model_training.framework.model.tokenizer.tokenizer_utils import text_to_token_ids, token_ids_to_text

In [26]:
config =load_model_config(dataset_max_len=1024)
# MODEL_NAME = 'gpt2'

model = return_model(MODEL_NAME)

# model = GPT.from_pretrained(config, "gpt2")
# print(f"Model Loaded: {model}")
# model.eval()
# model = model.to(device)

loading wights from pretrained gpt: gpt2-small
{'n_layer': 12, 'n_head': 12, 'emb_dim': 768}


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [27]:
# import inspect
# inspect.signature(tokenizer.encode)

In [28]:
model.eval()
text_1 = "Every effort moves you"
in_tokens = text_to_token_ids(text_1, tokenizer=tokenizer, 
                            #   allowed_special = {"<|endoftext|>"}
                              )
in_tokens = in_tokens.to(next(model.parameters()).device)
print(in_tokens.shape)  # [1, sequence_length]

torch.Size([1, 4])


In [29]:
out_tokens = gen_sample(model, in_tokens, max_new_tokens=40, context_len=1024)
print(token_ids_to_text(out_tokens, tokenizer))

Every effort moves you closer towards a goal. Think of goals in terms of steps: the ultimate goal may be the top of an enormous mountain, but the very first thing you do to start climbing is take your first step


### SFT-Training

In [30]:
import torch.nn.functional as F

model = model.to(device).eval()
sample_input = sample_input.to(device)
sample_target = sample_target.to(device)

with torch.no_grad():
    out = model(input_ids=sample_input)
    loss = F.cross_entropy(
        out.logits.reshape(-1, out.logits.size(-1)),
        sample_target.reshape(-1),
        ignore_index=-100,
    )

print(loss.item())

4.09375


In [31]:
from model_training.framework.train.core.train import train_model
from model_training.framework.train.core.config import TrainConfig
from model_training.framework.train.utils.optimizer import configure_optimizer
from model_training.framework.train.utils.lr_scheduler import cosine_lr

In [32]:
seq_length = 1024
total_batch_size = batch_size * seq_length
micro_batch_size = 4
grad_accum_steps = total_batch_size // (micro_batch_size * seq_length )
# seq_length = 1024
# micro_batch_size = train_loader.batch_size  # 1
# grad_accum_steps = 8
num_epochs = 2

In [33]:
from dataclasses import  dataclass
from typing import Any

@dataclass
class TrainComponents:
    optimizer:Any
    dataloader:Any
    scheduler:Any
    

In [34]:


train_config = TrainConfig(
    num_steps=(num_epochs * len(train_loader.loader)) // grad_accum_steps,
    learning_rate=1e-5,
    total_batch_size=micro_batch_size * seq_length * grad_accum_steps,
    micro_batch_size=micro_batch_size,
    seq_length=seq_length,
    accumulation_steps=grad_accum_steps,
    max_lr=1e-5,
    min_lr=1e-5,
    warmup_steps=0,
    weight_decay=0.1,
    device=device,
    use_compile=False,
    precision="fp32",
)

In [35]:
torch.manual_seed(123)

optimizer = configure_optimizer(model, train_config.weight_decay, train_config.learning_rate, master_process= True, device=train_config.device)

num decayed parameter tensors: 253, with 3,085,697,024 parameters
num non-decayed parameter tensors: 181, with 241,664 parameters
using fused AdamW: True


In [36]:
train_comp = TrainComponents(optimizer=optimizer , dataloader=train_loader, scheduler= cosine_lr)
train_comp

TrainComponents(optimizer=AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.95)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: True
    lr: 1e-05
    maximize: False
    weight_decay: 0.1

Parameter Group 1
    amsgrad: False
    betas: (0.9, 0.95)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: True
    lr: 1e-05
    maximize: False
    weight_decay: 0.0
), dataloader=<model_training.framework.data.post_train.dataloader.CyclingDataLoader object at 0x1553b5d87e00>, scheduler=<function cosine_lr at 0x1553a41c13a0>)

In [37]:
torch.set_float32_matmul_precision('high') ## TF16 -> Model Training 

In [38]:
print(train_config.accumulation_steps)

2


In [39]:
from datetime import datetime
from model_training.framework.train.tracking.logger import ExperimentLogger

run_name = datetime.now().strftime("%Y%m%d_%H%M%S")

logger = ExperimentLogger(
    project="sft-experiments",
    log_dir=str(project_root / "execution_results" / "runs" / run_name),
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/hice1/vchopra37/.netrc.
wandb: Currently logged in as: vedaangchopra (vedaangchopra_gatech) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [49]:
model = train_model(model, train_components=train_comp, train_config=train_config, logger = logger)

Current Device: cuda
Model set to Training mode:True
step 0 | loss: 4.156250 | norm :360.00 | dt: 4046.45 ms | tok/sec:4048.98


[W1101 03:53:15.952227206 CUDACachingAllocator.cpp:528] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 16580608, total: 42436526080).


step 1 | loss: 2.093750 | norm :146.00 | dt: 816.78 ms | tok/sec:20059.28
step 2 | loss: 1.445312 | norm :104.00 | dt: 525.36 ms | tok/sec:31186.40
step 3 | loss: 1.281250 | norm :97.00 | dt: 485.06 ms | tok/sec:33777.22
step 4 | loss: 0.921875 | norm :89.50 | dt: 496.91 ms | tok/sec:32971.65
step 5 | loss: 0.882812 | norm :68.00 | dt: 497.63 ms | tok/sec:32923.82


[W1101 03:53:17.392144112 CUDACachingAllocator.cpp:528] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 8192000, total: 42436526080).


step 6 | loss: 0.773438 | norm :33.50 | dt: 891.76 ms | tok/sec:18372.75


[W1101 03:53:18.547210469 CUDACachingAllocator.cpp:528] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 14483456, total: 42436526080).


... (89 outputs left out)

[W1101 03:54:23.575956677 CUDACachingAllocator.cpp:528] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 16580608, total: 42436526080).


step 110 | loss: 0.312500 | norm :4.34 | dt: 704.30 ms | tok/sec:23262.89
step 111 | loss: 0.357422 | norm :4.41 | dt: 512.94 ms | tok/sec:31941.61
step 112 | loss: 0.367188 | norm :4.38 | dt: 488.51 ms | tok/sec:33538.87
step 113 | loss: 0.404297 | norm :4.50 | dt: 474.75 ms | tok/sec:34510.83


[W1101 03:54:25.508345312 CUDACachingAllocator.cpp:528] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 14483456, total: 42436526080).


step 114 | loss: 0.308594 | norm :4.12 | dt: 894.57 ms | tok/sec:18314.96
step 115 | loss: 0.359375 | norm :4.53 | dt: 509.78 ms | tok/sec:32139.24


In [53]:
# %tensorboard --logdir "{project_root}/execution_results/runs"

In [ ]:
# # text_1 = "What is your name ? "
# text_1 = "Every effort moves you"
# in_tokens = text_to_token_ids(text_1, tokenizer=tokenizer, 
#                             #   allowed_special = {"<|endoftext|>"}
#                               )
# in_tokens=in_tokens.to(device)
# print(in_tokens.shape)  # [1, sequence_length]


# out_tokens = gen_sample(model, in_tokens, max_new_tokens=40, context_len=1024)
# print(token_ids_to_text(out_tokens, tokenizer))

### Setting up LORA 

In [41]:
q_proj = model.model.layers[0].self_attn.q_proj

print("Layer:", q_proj)
print("Weight shape:", tuple(q_proj.weight.shape))
print("Bias shape:", tuple(q_proj.bias.shape) if q_proj.bias is not None else None)
print("Trainable parameters:", sum(p.numel() for p in q_proj.parameters() if p.requires_grad))

Layer: Linear(in_features=2048, out_features=2048, bias=True)
Weight shape: (2048, 2048)
Bias shape: (2048,)
Trainable parameters: 4196352


In [73]:
import torch
import torch.nn as nn

rank, alpha = 4, 4
A = nn.Parameter(torch.randn(rank, q_proj.in_features))
B = nn.Parameter(torch.zeros(q_proj.out_features, rank))

print(A.shape, B.shape, (B @ A).shape)

torch.Size([4, 2048]) torch.Size([2048, 4]) torch.Size([2048, 2048])


In [74]:
x = torch.randn(1, 2048)
x.shape

torch.Size([1, 2048])

In [75]:
low_rank = x @ A.T
adjustment = low_rank @B.T

In [76]:
print("Low-rank shape:", low_rank.shape)
print("Adjustment shape:", adjustment.shape)
print("Adjustment is zero:", torch.all(adjustment == 0).item())

Low-rank shape: torch.Size([1, 4])
Adjustment shape: torch.Size([1, 2048])
Adjustment is zero: True


In [77]:
alpha =4 
x = x.to(device = q_proj.weight.device, dtype = q_proj.weight.dtype)

In [78]:
base_output = q_proj(x)

In [79]:
adjustment = (x @A.to(x).T) @ B.to(x).T
lora_output = base_output  + (alpha /rank) * adjustment

print(lora_output.shape)
print(torch.allclose(base_output, lora_output))

torch.Size([1, 2048])
True


### Lora Linear Layer 

In [80]:
class LoRALinear(nn.Module):
    def __init__(self, base_layer:nn.Linear, rank, alpha):
        super().__init__()
        self.base_layer = base_layer 
        self.base_layer.requires_grad_(False)
        
        self.A = nn.Linear(base_layer.in_features, rank, bias=False)
        self.B = nn.Linear(rank, base_layer.out_features, bias=False)
        
        nn.init.kaiming_uniform_(self.A.weight, a=5**0.5)
        nn.init.zeros_(self.B.weight)
        self.scaling = alpha / rank
        
    def forward(self, x):
        return self.base_layer(x) + self.scaling * self.B(self.A(x))
        
toy = nn.Linear(8, 6)
lora = LoRALinear(toy, rank, alpha)

print(lora.base_layer.weight.requires_grad)
print(lora.base_layer.bias.requires_grad)

False
False


In [81]:
x = torch.randn(1, 8)
print(torch.allclose(lora(x), lora.base_layer(x)))

True


In [82]:
lora.zero_grad()
lora(x).sum().backward()

print("Base:", lora.base_layer.weight.grad)
print("A gradient norm:", lora.A.weight.grad.norm().item())
print("B gradient norm:", lora.B.weight.grad.norm().item())

Base: None
A gradient norm: 0.0
B gradient norm: 0.9065287709236145


In [83]:
optimizer = torch.optim.SGD([lora.A.weight, lora.B.weight], lr=0.01)

b_before = lora.B.weight.detach().clone()
optimizer.step()

print("B changed:", not torch.equal(b_before, lora.B.weight))
print("Output still matches base:", torch.allclose(lora(x), lora.base_layer(x)))

B changed: True
Output still matches base: False


In [84]:
print(model.model.layers[0])

Qwen2DecoderLayer(
  (self_attn): Qwen2Attention(
    (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
    (k_proj): Linear(in_features=2048, out_features=256, bias=True)
    (v_proj): Linear(in_features=2048, out_features=256, bias=True)
    (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
  )
  (mlp): Qwen2MLP(
    (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
    (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
    (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
  (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
)


In [86]:
q_proj = model.model.layers[0].self_attn.q_proj
q_lora = LoRALinear(q_proj, rank=4, alpha=4).to(q_proj.weight)
print(q_lora)

LoRALinear(
  (base_layer): Linear(in_features=2048, out_features=2048, bias=True)
  (A): Linear(in_features=2048, out_features=4, bias=False)
  (B): Linear(in_features=4, out_features=2048, bias=False)
)


In [87]:
model.model.layers[0].self_attn.q_proj = q_lora

print(model.model.layers[0].self_attn.q_proj)

LoRALinear(
  (base_layer): Linear(in_features=2048, out_features=2048, bias=True)
  (A): Linear(in_features=2048, out_features=4, bias=False)
  (B): Linear(in_features=4, out_features=2048, bias=False)
)


In [88]:
model.requires_grad_(False)
q_lora.A.requires_grad_(True)
q_lora.B.requires_grad_(True)

for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.shape)

model.layers.0.self_attn.q_proj.A.weight torch.Size([4, 2048])
model.layers.0.self_attn.q_proj.B.weight torch.Size([2048, 4])


In [89]:
attn = model.model.layers[0].self_attn
v_proj = attn.v_proj

attn.v_proj = LoRALinear(v_proj, rank=4, alpha=4).to(v_proj.weight)

print(attn.v_proj)

LoRALinear(
  (base_layer): Linear(in_features=2048, out_features=256, bias=True)
  (A): Linear(in_features=2048, out_features=4, bias=False)
  (B): Linear(in_features=4, out_features=256, bias=False)
)
